# GazeSpeakZero — E1b re-analysis (v3: webcam–Tobii clock correction), about 40–70 min

This notebook only re-runs the **analysis** on the features the first run already saved to Google Drive (`MyDrive/gazespeakzero/E1b/features`). It needs no download and no video processing. Run the cells one at a time with Shift + Enter.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/gazespeakzero/E1b'
import os, glob
print(len(glob.glob(f'{WORK}/features/P_*_frames.csv')), 'participant feature files found (expect 51)')

In [ ]:
!mkdir -p /content/e1 /content/e1b

In [ ]:
%%writefile /content/e1/analyze.py
"""
E1, step 2: calibration-free gaze-zone accuracy on MPIIFaceGaze.

Reads the per-subject CSVs from extract_features.py and writes tables,
figures and a markdown report to --out.

    python analyze.py --features features --out results

Protocol
--------
* Ground truth: the on-screen target (pixels) divided by that subject's
  screen size gives a position in [0, 1]^2; the zone is its grid cell.
* Every learned method is leave-one-subject-out (LOSO): trained on 14 people,
  tested on the 15th, so no calibration data from the test person is used.
* A frame with no detected face counts as WRONG in every accuracy figure.
* Headline = mean of the 15 per-subject accuracies, with a 95 % bootstrap CI
  over subjects. The pooled frame-level figure is reported alongside.
* Calibration reference: k frames of the test person correct the M2 output
  (k < 5: offset only; k >= 5: per-axis least-squares slope and offset,
  slope clipped to [0.5, 3]). Two draws of the k frames:
    - "first session": from the person's first recording day, scored on
      the other days (how a real calibration is used);
    - "random": from any day, scored on the rest.
  20 draws per k.
"""
from __future__ import annotations

import argparse
import glob
import json
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

GRIDS = [(1, 2), (2, 1), (1, 3), (2, 2), (3, 3), (4, 4)]
CALIB_K = [1, 3, 5, 9, 16]
N_CALIB_DRAWS = 20
N_BOOT = 10000
SEED = 0

EYE = []
for e in ("reye", "leye"):
    EYE += [f"{e}_h", f"{e}_v", f"{e}_vlid", f"{e}_open"]
BS = ["bs_eyeLookInLeft", "bs_eyeLookInRight", "bs_eyeLookOutLeft", "bs_eyeLookOutRight",
      "bs_eyeLookUpLeft", "bs_eyeLookUpRight", "bs_eyeLookDownLeft", "bs_eyeLookDownRight",
      "bs_eyeBlinkLeft", "bs_eyeBlinkRight", "bs_eyeSquintLeft", "bs_eyeSquintRight"]
HEAD = ([f"R{i}{j}" for i in range(3) for j in range(3)]
        + ["tx", "ty", "tz", "yaw", "pitch", "roll", "nose_x", "nose_y", "interocular_norm"])
FEATURE_SETS = {
    "M0-fit": ["m0_cx", "m0_cy"],
    "M1": EYE + BS,
    "M2": EYE + BS + HEAD,
}


# ----------------------------------------------------------------------------
def load(features_dir: str) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(features_dir, "p[0-9][0-9].csv")))
    if not files:
        raise SystemExit(f"no pNN.csv in {features_dir}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    counts = df.groupby("subject").size()
    print("frames per subject:", counts.to_dict(), flush=True)
    if counts.min() < 0.9 * counts.max():
        print("WARNING: some subjects have far fewer frames than others", flush=True)
    df["day"] = df["file"].str.split("/").str[0]
    df["gx"] = df["gt_px_x"] / df["screen_w_px"]
    df["gy"] = df["gt_px_y"] / df["screen_h_px"]
    df["detected"] = df["detected"].fillna(0).astype(int)
    return df


def zone_of(x, y, rows, cols):
    """Grid cell of normalised screen positions; NaN in -> -1 (always wrong)."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    bad = ~(np.isfinite(x) & np.isfinite(y))
    c = np.clip(np.floor(np.nan_to_num(x) * cols), 0, cols - 1).astype(int)
    r = np.clip(np.floor(np.nan_to_num(y) * rows), 0, rows - 1).astype(int)
    z = r * cols + c
    z[bad] = -1
    return z


def m0_original_phi(cx, cy):
    """Repo's _phi for (3,3) with theta = 0.333, verbatim."""
    cx = np.asarray(cx, float)
    cy = np.asarray(cy, float)
    bad = ~(np.isfinite(cx) & np.isfinite(cy))
    col = np.clip(np.floor(np.nan_to_num(cx) / 0.333), 0, 2).astype(int)
    row = np.clip(np.floor(np.nan_to_num(cy) / 0.333), 0, 2).astype(int)
    z = row * 3 + col
    z[bad] = -1
    return z


def make_model(kind: str):
    if kind == "hgb":
        return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                             early_stopping=False, random_state=SEED)
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.0))


def loso_predict(df: pd.DataFrame, feats: list[str], kind: str) -> np.ndarray:
    """Returns an (N, 2) array of predicted (x, y); NaN where no face."""
    pred = np.full((len(df), 2), np.nan)
    det = df["detected"].values == 1
    for s in df["subject"].unique():
        tr = (df["subject"].values != s) & det
        te = (df["subject"].values == s) & det
        if te.sum() == 0:
            continue
        X_tr, X_te = df.loc[tr, feats].values, df.loc[te, feats].values
        for j, tgt in enumerate(("gx", "gy")):
            m = make_model(kind)
            m.fit(X_tr, df.loc[tr, tgt].values)
            pred[te, j] = m.predict(X_te)
    return pred


def per_subject_acc(df, zpred, rows, cols) -> pd.Series:
    zt = zone_of(df["gx"], df["gy"], rows, cols)
    ok = pd.Series(zpred == zt, index=df.index)
    return ok.groupby(df["subject"]).mean()


def boot_ci(v: np.ndarray, rng) -> tuple[float, float]:
    """BCa bootstrap CI of the mean over subjects; percentile if BCa is undefined."""
    from scipy.stats import bootstrap
    v = np.asarray(v, float)
    if np.allclose(v, v[0]):
        return float(v[0]), float(v[0])
    for method in ("BCa", "percentile"):
        r = bootstrap((v,), np.mean, n_resamples=N_BOOT, method=method, random_state=rng)
        lo, hi = r.confidence_interval
        if np.isfinite(lo) and np.isfinite(hi):
            return float(lo), float(hi)
    return float("nan"), float("nan")


def balanced_acc(zt, zp, K) -> float:
    return float(np.mean([np.mean(zp[zt == k] == k) for k in range(K) if np.any(zt == k)]))


def mm_error(df, pred) -> pd.Series:
    dx = (pred[:, 0] - df["gx"].values) * df["screen_w_mm"].values
    dy = (pred[:, 1] - df["gy"].values) * df["screen_h_mm"].values
    return pd.Series(np.sqrt(dx ** 2 + dy ** 2), index=df.index)


def fit_correction(p, t, k):
    if k < 5:
        return 1.0, float(np.mean(t - p))
    pm, tm = p.mean(), t.mean()
    sxx = np.sum((p - pm) ** 2)
    a = float(np.clip(np.sum((p - pm) * (t - tm)) / sxx, 0.5, 3.0)) if sxx > 1e-9 else 1.0
    return a, float(tm - a * pm)


def calibrate(df, base_pred, rows_cols, rng) -> list[dict]:
    """Per-subject k-shot correction of base_pred, scored on frames not used to calibrate."""
    out = []
    det = df["detected"].values == 1
    days = df["day"].values
    for s in df["subject"].unique():
        idx_all = np.where(df["subject"].values == s)[0]
        first_day = sorted(set(days[idx_all]))[0]
        pools = {
            "random": (idx_all[det[idx_all]], None),
            "first session": (idx_all[det[idx_all] & (days[idx_all] == first_day)],
                              idx_all[days[idx_all] != first_day]),
        }
        for scheme, (pool, test_fixed) in pools.items():
            for k in CALIB_K:
                if len(pool) < k or (test_fixed is not None and len(test_fixed) == 0):
                    continue
                for d in range(N_CALIB_DRAWS):
                    cal = rng.choice(pool, size=k, replace=False)
                    test = np.setdiff1d(idx_all, cal) if test_fixed is None else test_fixed
                    corr = np.empty((len(test), 2))
                    for j, tgt in enumerate(("gx", "gy")):
                        a, b = fit_correction(base_pred[cal, j], df[tgt].values[cal], k)
                        corr[:, j] = a * base_pred[test, j] + b   # NaN (no face) stays NaN -> wrong
                    sub = df.iloc[test]
                    for (r, c) in rows_cols:
                        acc = float(np.mean(zone_of(corr[:, 0], corr[:, 1], r, c)
                                            == zone_of(sub["gx"], sub["gy"], r, c)))
                        out.append({"scheme": scheme, "subject": s, "k": k, "draw": d,
                                    "grid": f"{r}x{c}", "acc": acc})
    return out


def confusion_png(df, zpred, rows, cols, title, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    K = rows * cols
    zt = zone_of(df["gx"], df["gy"], rows, cols)
    M = np.zeros((K, K + 1))
    for t, p in zip(zt, zpred):
        M[t, p if p >= 0 else K] += 1
    M = M / M.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(0.55 * K + 2.5, 0.55 * K + 1.8))
    im = ax.imshow(M, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(K + 1), [str(i) for i in range(K)] + ["no face"], rotation=45)
    ax.set_yticks(range(K))
    ax.set_xlabel("predicted zone")
    ax.set_ylabel("true zone")
    ax.set_title(title, fontsize=9)
    for i in range(K):
        for j in range(K + 1):
            if M[i, j] >= 0.005:
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if M[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


# ----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--features", default="features")
    ap.add_argument("--out", default="results")
    ap.add_argument("--skip-calibration", action="store_true")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    rng = np.random.default_rng(SEED)

    df = load(args.features)
    subjects = sorted(df["subject"].unique())
    det = df["detected"].values == 1

    # ---- data QA -------------------------------------------------------------
    qa = {
        "n_subjects": len(subjects),
        "n_frames": int(len(df)),
        "detection_rate": float(det.mean()),
        "detection_rate_min_subject": float(df.groupby("subject")["detected"].mean().min()),
        "gt_outside_screen_frac": float(((df.gx < 0) | (df.gx >= 1) | (df.gy < 0) | (df.gy >= 1)).mean()),
        "median_landmark_ms": float(df["landmark_ms"].median()),
        "unreadable_images": int(df["read_error"].fillna(0).sum()),
        "frames_per_subject": {k: int(v) for k, v in df.groupby("subject").size().items()},
        "median_R_scale": float(df.loc[det, "R_scale"].median()) if "R_scale" in df else None,
        "median_reye_h": float(df.loc[det, "reye_h"].median()),
    }
    # MediaPipe eye corners vs the dataset's hand-annotated eye corners (first 4 landmarks)
    if det.any():
        d = df[det]
        mp_c = np.stack([d[[f"{k}_x", f"{k}_y"]].values for k in ("mp_rout", "mp_rin", "mp_lout", "mp_lin")], 1)
        an_c = np.stack([d[[f"ann_lm{i}_x", f"ann_lm{i}_y"]].values for i in range(4)], 1)
        dist = np.linalg.norm(an_c[:, :, None, :] - mp_c[:, None, :, :], axis=-1).min(axis=2).mean(axis=1)
        io = d["interocular_norm"].values * d["img_w"].values
        rel = dist / np.maximum(io, 1e-6)
        qa["eye_corner_err_median_rel_interocular"] = float(np.median(rel))
        qa["eye_corner_err_gt_0.2_frac"] = float(np.mean(rel > 0.2))

    # ---- M0 diagnostics: where does the original feature actually land? ---------
    m0_zone = m0_original_phi(df["m0_cx"], df["m0_cy"])
    m0_hist = pd.Series(m0_zone).value_counts(normalize=True).sort_index()
    qa["m0_zone_distribution"] = {int(k): round(float(v), 4) for k, v in m0_hist.items()}
    qa["m0_cx_quantiles"] = df.loc[det, "m0_cx"].quantile([0.01, 0.5, 0.99]).round(4).tolist()
    qa["m0_cy_quantiles"] = df.loc[det, "m0_cy"].quantile([0.01, 0.5, 0.99]).round(4).tolist()
    true33 = zone_of(df["gx"], df["gy"], 3, 3)
    qa["gt_zone_distribution_3x3"] = {int(k): round(float(v), 4)
                                      for k, v in pd.Series(true33).value_counts(normalize=True).sort_index().items()}

    # ---- predictions -----------------------------------------------------------
    preds = {}
    preds["M0 original"] = np.c_[df["m0_cx"], df["m0_cy"]]
    preds["M0 original (x flipped)"] = np.c_[1 - df["m0_cx"], df["m0_cy"]]
    for name, feats in FEATURE_SETS.items():
        for kind in ("ridge", "hgb"):
            print(f"LOSO {name} {kind} ...", flush=True)
            preds[f"{name} {kind}"] = loso_predict(df, feats, kind)

    # majority zone of the training subjects, per grid (LOSO)
    rows_out, per_subj_rows = [], []
    for (r, c) in GRIDS:
        g = f"{r}x{c}"
        zt = zone_of(df["gx"], df["gy"], r, c)
        maj = np.full(len(df), -1)
        for s in subjects:
            tr = df["subject"].values != s
            maj[~tr] = pd.Series(zt[tr]).mode().iloc[0]
        maj = np.where(det, maj, -1)                       # needs a face like every other method
        method_z = {"chance (1/K)": None, "majority zone": maj}
        for name, p in preds.items():
            if name.startswith("M0 original") and (r, c) == (3, 3):
                z = m0_original_phi(p[:, 0], p[:, 1])
            else:
                z = zone_of(p[:, 0], p[:, 1], r, c)
            if name.startswith("M0 original"):
                z = np.where(det, z, -1)
                if (r, c) != (3, 3):
                    name = name.replace("M0 original", "M0 generalised")
            method_z[name] = z
        for name, z in method_z.items():
            if z is None:
                rows_out.append({"grid": g, "method": name, "acc_mean_subj": 1 / (r * c),
                                 "ci_lo": np.nan, "ci_hi": np.nan, "acc_pooled": 1 / (r * c),
                                 "acc_detected_only": 1 / (r * c), "balanced_acc": 1 / (r * c),
                                 "acc_min_subj": np.nan, "acc_max_subj": np.nan})
                continue
            ps = per_subject_acc(df, z, r, c)
            lo, hi = boot_ci(ps.values, rng)
            rows_out.append({"grid": g, "method": name, "acc_mean_subj": ps.mean(),
                             "ci_lo": lo, "ci_hi": hi, "acc_pooled": float(np.mean(z == zt)),
                             "acc_detected_only": float(np.mean(z[det] == zt[det])),
                             "balanced_acc": balanced_acc(zt, z, r * c),
                             "acc_min_subj": ps.min(), "acc_max_subj": ps.max()})
            for s, a in ps.items():
                per_subj_rows.append({"grid": g, "method": name, "subject": s, "acc": a})
        if (r, c) in [(3, 3), (2, 2), (1, 3)]:
            m0 = "M0 original" if (r, c) == (3, 3) else "M0 generalised"
            for name, fname in ((m0, "M0"), ("M2 hgb", "M2_hgb")):
                confusion_png(df, method_z[name], r, c, f"{name}, {g} (rows: true, cols: predicted)",
                              os.path.join(args.out, f"confusion_{fname}_{g}.png"))

    acc = pd.DataFrame(rows_out)
    acc.to_csv(os.path.join(args.out, "e1_zone_accuracy.csv"), index=False)
    pd.DataFrame(per_subj_rows).to_csv(os.path.join(args.out, "e1_per_subject.csv"), index=False)

    mm = []
    for name, p in preds.items():
        if name.startswith("M0"):
            continue
        e = mm_error(df, p)[det]
        mm.append({"method": name, "median_mm": e.median(), "mean_mm": e.mean(),
                   "p90_mm": e.quantile(0.9)})
    mm = pd.DataFrame(mm)
    mm.to_csv(os.path.join(args.out, "e1_mm_error.csv"), index=False)

    cal = None
    if not args.skip_calibration:
        print("calibration reference ...", flush=True)
        cal = pd.DataFrame(calibrate(df, preds["M2 hgb"], [(1, 3), (2, 2), (3, 3), (4, 4)], rng))
        cal.to_csv(os.path.join(args.out, "e1_calibration_raw.csv"), index=False)
        cal = (cal.groupby(["scheme", "grid", "k", "subject"])["acc"].mean()
                  .groupby(["scheme", "grid", "k"]).agg(["mean", "min", "max"]).reset_index())
        cal.to_csv(os.path.join(args.out, "e1_calibration.csv"), index=False)

    with open(os.path.join(args.out, "e1_qa.json"), "w") as f:
        json.dump(qa, f, indent=2)

    # ---- report ------------------------------------------------------------------
    L = ["# E1: calibration-free gaze-zone accuracy on MPIIFaceGaze", "",
         f"{qa['n_subjects']} subjects, {qa['n_frames']} frames. Face detected in "
         f"{qa['detection_rate']:.1%} of frames (lowest subject {qa['detection_rate_min_subject']:.1%}); "
         "undetected frames count as wrong.", "",
         "Accuracy = mean of per-subject accuracies (95% bootstrap CI over subjects). "
         "All learned methods are leave-one-subject-out.", ""]
    for g in [f"{r}x{c}" for r, c in GRIDS]:
        L += [f"## Grid {g}", "",
              "| Method | Accuracy | 95% CI | Pooled | Face-found frames only | Balanced | Worst subject | Best subject |",
              "|---|---|---|---|---|---|---|---|"]
        for _, x in acc[acc.grid == g].iterrows():
            ci = "" if np.isnan(x.ci_lo) else f"{x.ci_lo:.3f}–{x.ci_hi:.3f}"
            mn = "" if np.isnan(x.acc_min_subj) else f"{x.acc_min_subj:.3f}"
            mx = "" if np.isnan(x.acc_max_subj) else f"{x.acc_max_subj:.3f}"
            L.append(f"| {x.method} | {x.acc_mean_subj:.3f} | {ci} | {x.acc_pooled:.3f} | "
                     f"{x.acc_detected_only:.3f} | {x.balanced_acc:.3f} | {mn} | {mx} |")
        L.append("")
    L += ["## On-screen error (detected frames, mm)", "", "| Method | Median | Mean | 90th pct |",
          "|---|---|---|---|"]
    for _, x in mm.iterrows():
        L.append(f"| {x.method} | {x.median_mm:.0f} | {x.mean_mm:.0f} | {x.p90_mm:.0f} |")
    if cal is not None:
        L += ["", "## With k calibration frames from the test person (M2 hgb + correction)", "",
              "k < 5: offset only; k >= 5: slope and offset per axis. "
              "'first session' calibrates on day 1 and scores the other days.", "",
              "| Scheme | Grid | k | Mean acc | Worst subject | Best subject |", "|---|---|---|---|---|---|"]
        for _, x in cal.iterrows():
            L.append(f"| {x.scheme} | {x.grid} | {x.k} | {x['mean']:.3f} | {x['min']:.3f} | {x['max']:.3f} |")
    L += ["", "## Data QA and M0 diagnostics", "", "```", json.dumps(qa, indent=2), "```"]
    with open(os.path.join(args.out, "E1_REPORT.md"), "w") as f:
        f.write("\n".join(L) + "\n")
    print("\n".join(L))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e1b/eott_analyze.py
"""
E1b, step 2: does averaging over a dwell remove the frame-level scatter found in E1?

Inputs: the per-participant frame and Tobii CSVs written by eott_extract.py.

Protocol (fixed before seeing any EOTT result)
----------------------------------------------
* Ground truth comes from the Tobii Pro X3-120 (120 Hz), not from on-screen dots.
* v2 (21 Sep, after v1 found only 106 windows): dwell windows are consecutive
  non-overlapping windows of length L (0.5 s, 1 s) over each Tobii stream,
  kept when >= 70% of samples are valid and the gaze is stable,
  sqrt(var x + var y) <= 0.03 of the screen (primary; 0.02 and 0.05 reported
  as sensitivity). The window's true position is the mean of its Tobii samples.
* v2: the per-frame model is trained on every labelled frame of the other
  participants (v1 used only frames inside windows: 1,537 frames).
* Per-frame gaze model: the E1 "M2" feature set, gradient boosting, trained with
  10-fold GroupKFold over participants (no participant is ever in both train and
  test).
  Laptop and PC participants are pooled, with setup as a feature.
* A window's prediction from its frames, in four ways:
    single = the middle frame only;
    mean / median = mean or median of the per-frame predicted positions;
    vote = majority of the per-frame zones.
  Frames without a face are skipped. A window with fewer than
  MIN_FRAME_FRAC of the expected frames is WRONG (the system would not
  have answered).
* Accuracy = mean of per-participant accuracies, BCa 95% CI over participants.
* How much error is shared within a dwell: for each window, per-frame errors
  e_t are split into the window mean (shared) and deviations (independent).
  shared_frac = var(window means) / var(all frame errors). If this is near 1,
  averaging cannot help.
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(HERE, "..", "e1"))
from analyze import EYE, BS, HEAD, zone_of, boot_ci   # noqa: E402

GRIDS = [(1, 2), (2, 1), (1, 3), (2, 2), (3, 3), (4, 4)]
DWELLS_MS = [500, 1000]
DISP = 0.04            # v1 (I-DT), kept for the record
DISP_STD = [0.03, 0.02, 0.05]   # primary first, then sensitivity
MARGIN_MS = 150
MIN_FRAME_FRAC = 0.5
N_FOLDS = 10
SEED = 0
FEATS = EYE + BS + HEAD + ["is_laptop"]


def load(d: str):
    fr = sorted(glob.glob(os.path.join(d, "P_*_frames.csv")))
    tb = sorted(glob.glob(os.path.join(d, "P_*_tobii.csv")))
    if not fr:
        raise SystemExit(f"no P_*_frames.csv in {d}")
    F = pd.concat([pd.read_csv(f) for f in fr], ignore_index=True)
    T = pd.concat([pd.read_csv(f) for f in tb], ignore_index=True)
    F["detected"] = F["detected"].fillna(0).astype(int)
    F["is_laptop"] = (F["setup"].str.lower() == "laptop").astype(int)
    return F, T


def fixations(t: pd.DataFrame, min_ms: float) -> list[tuple[float, float]]:
    """Dispersion-threshold (I-DT) fixations on one task's Tobii stream.
    Returns (start_ms, end_ms) for every fixation lasting at least min_ms."""
    t = t[t.tvalid == 1].dropna(subset=["tx", "ty"])
    ts, xs, ys = t.t_ms.values, t.tx.values, t.ty.values
    n = len(ts)

    def disp(i, j):
        return (xs[i:j + 1].max() - xs[i:j + 1].min()) + (ys[i:j + 1].max() - ys[i:j + 1].min())

    out, i = [], 0
    while i < n:
        j = np.searchsorted(ts, ts[i] + min_ms)          # first sample at or beyond min_ms
        if j >= n:
            break
        if disp(i, j) <= DISP and np.diff(ts[i:j + 1]).max(initial=0) <= 50:
            while j + 1 < n and ts[j + 1] - ts[j] < 50 and disp(i, j + 1) <= DISP:
                j += 1
            out.append((float(ts[i]), float(ts[j])))
            i = j + 1
        else:
            i += 1
    return out


def make_windows(T: pd.DataFrame, L: float, disp_std: float) -> pd.DataFrame:
    """Non-overlapping windows of length L laid end to end over each video's Tobii
    stream. A window is a dwell if >= 70% of its Tobii samples are valid and the
    gaze is stable: sqrt(var x + var y) <= disp_std (normalised screen units).
    (v1 required a whole I-DT fixation of L + 300 ms; on EOTT that left only 106
    windows of 0.5 s in 51 people, too few to estimate anything.)"""
    out = []
    for (p, task, video), t in T.groupby(["participant", "task", "video"]):
        t = t.sort_values("t_ms")
        t0 = t.t_ms.min()
        k = ((t.t_ms - t0) // L).astype(int)
        v = t[t.tvalid == 1]
        g = v.groupby(k[v.index])
        st = pd.DataFrame({"n": g.size(), "gx": g.tx.mean(), "gy": g.ty.mean(),
                           "sx": g.tx.std(), "sy": g.ty.std()})
        st = st[(st.n >= 0.7 * L / 1000 * 120) & (np.sqrt(st.sx ** 2 + st.sy ** 2) <= disp_std)]
        for kk, r in st.iterrows():
            out.append({"participant": p, "task": task, "video": video, "w_start": t0 + kk * L,
                        "w_end": t0 + (kk + 1) * L, "gx": r.gx, "gy": r.gy})
    cols = ["participant", "task", "video", "w_start", "w_end", "gx", "gy"]
    W = pd.DataFrame(out, columns=cols)
    return W[np.isfinite(W.gx) & np.isfinite(W.gy)].sort_values(["participant", "video", "w_start"]).reset_index(drop=True)


def assign_frames(F: pd.DataFrame, W: pd.DataFrame) -> pd.Series:
    """window id for each frame (or -1)."""
    wid = pd.Series(-1, index=F.index)
    for (p, video), w in W.groupby(["participant", "video"]):
        m = (F.participant == p) & (F.video == video)
        e = F.loc[m, "epoch_ms"].values
        k = np.searchsorted(w.w_start.values, e, side="right") - 1
        inside = (k >= 0) & (e < w.w_end.values[np.clip(k, 0, len(w) - 1)])
        wid.loc[m] = np.where(inside, w.index.values[np.clip(k, 0, len(w) - 1)], -1)
    return wid


def fit_predict(F: pd.DataFrame, train_mask: np.ndarray, n_folds: int = N_FOLDS) -> np.ndarray:
    pred = np.full((len(F), 2), np.nan)
    det = F.detected.values == 1
    groups = F.participant.values
    n_groups = len(np.unique(groups))
    gkf = GroupKFold(n_splits=min(n_folds, n_groups))
    X = F[FEATS].values
    for k, (tr, te) in enumerate(gkf.split(X, groups=groups)):
        trm = np.zeros(len(F), bool); trm[tr] = True
        trm &= train_mask & det
        tem = np.zeros(len(F), bool); tem[te] = True
        tem &= det
        for j, tgt in enumerate(("tobii_x", "tobii_y")):
            m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                              early_stopping=False, random_state=SEED)
            m.fit(X[trm], F[tgt].values[trm])
            pred[tem, j] = m.predict(X[tem])
        print(f"  fold {k + 1}/{gkf.get_n_splits()}", flush=True)
    return pred


def window_predictions(F, pred, W, L, fps_by_video):
    out = {}
    Fw = F.assign(px=pred[:, 0], py=pred[:, 1])
    Fw = Fw[Fw.wid >= 0]
    g = Fw.groupby("wid")
    for method in ("single", "mean", "median", "vote"):
        out[method] = {}
    expected = {i: max(1.0, fps_by_video.get(v, 30.0) * L / 1000) for i, v in zip(W.index, W.video)}
    centre = ((W.w_start + W.w_end) / 2).to_dict()
    for wid, d in g:
        d = d[d.detected == 1]
        if len(d) < MIN_FRAME_FRAC * expected[wid]:
            continue
        mid = d.iloc[int(np.argmin(np.abs(d.epoch_ms.values - centre[wid])))]
        out["single"][wid] = (mid.px, mid.py)
        out["mean"][wid] = (d.px.mean(), d.py.mean())
        out["median"][wid] = (d.px.median(), d.py.median())
        out["vote"][wid] = d[["px", "py"]].values          # zone vote done per grid
    return out


def score_windows(F, pred, W, L, fps_by_video, rng, setup_of=None):
    """Accuracy per grid and aggregation method, plus the shared-error ICC."""
    rows, shared = [], []
    F = F.copy()
    F["wid"] = assign_frames(F, W)
    wp = window_predictions(F, pred, W, L, fps_by_video)
    for (r, c) in GRIDS:
        zt = pd.Series(zone_of(W.gx, W.gy, r, c), index=W.index)
        for method, d in wp.items():
            zp = pd.Series(-1, index=W.index)          # unanswered windows are wrong
            for wid, v in d.items():
                if method == "vote":
                    z = zone_of(v[:, 0], v[:, 1], r, c)
                    z = z[z >= 0]
                    if len(z):
                        cnt = np.bincount(z, minlength=r * c)
                        top = np.flatnonzero(cnt == cnt.max())
                        zp[wid] = top[0] if len(top) == 1 else zone_of([v[:, 0].mean()], [v[:, 1].mean()], r, c)[0]
                else:
                    zp[wid] = zone_of([v[0]], [v[1]], r, c)[0]
            ok = ((zp == zt) & (zp >= 0) & (zt >= 0)).groupby(W.participant).mean()
            lo, hi = boot_ci(ok.values, rng)
            su = ok.groupby(ok.index.map(setup_of)).mean() if setup_of else {}
            rows.append({"grid": f"{r}x{c}", "method": method, "acc": ok.mean(), "ci_lo": lo, "ci_hi": hi,
                         "worst": ok.min(), "best": ok.max(), "answered_frac": len(d) / max(len(W), 1),
                         "acc_laptop": su.get("Laptop", np.nan), "acc_pc": su.get("PC", np.nan)})
    d = F[(F.wid >= 0) & (F.detected == 1)].join(W[["gx", "gy"]], on="wid")
    for ax, g in (("x", "gx"), ("y", "gy")):
        e = d[f"p{ax}"] - d[g]
        grp = e.groupby(d.wid)
        wm = grp.transform("mean")
        k, N = grp.ngroups, len(e)
        n_i = grp.size().values
        msb = float((n_i * (grp.mean().values - e.mean()) ** 2).sum() / max(k - 1, 1))
        msw = float(((e - wm) ** 2).sum() / max(N - k, 1))
        n0 = (N - (n_i ** 2).sum() / N) / max(k - 1, 1)
        icc = (msb - msw) / (msb + (n0 - 1) * msw) if (msb + (n0 - 1) * msw) > 0 else np.nan
        shared.append({"axis": ax, "shared_frac_icc": float(icc), "shared_frac_naive": float(wm.var() / e.var()),
                       "frames_per_window_mean": float(n_i.mean()),
                       "rmse_frame": float(np.sqrt((e ** 2).mean())),
                       "rmse_window_mean": float(np.sqrt((grp.mean() ** 2).mean()))})
    return rows, shared


def lag_check(F, T, pred):
    """Per participant: lag (ms) at which predicted x best matches Tobii x.
    Near 0 means webcam frames and Tobii samples are correctly aligned in time."""
    out = []
    Fp = F.assign(px=pred[:, 0])
    for p, d in Fp[Fp.px.notna()].groupby("participant"):
        best = (np.nan, -np.inf)
        tp = T[(T.participant == p) & (T.tvalid == 1)].dropna(subset=["tx"])
        rs = {}
        for v, dv in d.groupby("video"):
            tv = tp[tp.video == v].sort_values("t_ms")
            dv = dv.sort_values("epoch_ms")
            if len(tv) < 100 or len(dv) < 100:
                continue
            for lag in range(-600, 601, 33):
                j = np.interp(dv.epoch_ms.values + lag, tv.t_ms.values, tv.tx.values)
                rs.setdefault(lag, []).append(np.corrcoef(dv.px.values, j)[0, 1])
        if rs:
            lag, r = max(((l, float(np.nanmean(v))) for l, v in rs.items()), key=lambda z: z[1])
            r0 = float(np.nanmean(rs[min(rs, key=abs)]))
            out.append({"participant": p, "best_lag_ms": lag, "r_at_best": r, "r_at_zero": r0})
    return pd.DataFrame(out)


def estimate_lags(F, T, signal, step=10, span=800):
    """Clock offset between each participant's webcam frames and Tobii: the lag at
    which `signal` (a per-frame horizontal gaze signal from the webcam) correlates
    best with Tobii x, averaged over that participant's videos. Positive lag = the
    frame shows the eye at epoch_ms + lag. Only the Tobii time series is used, as
    in any two-sensor clock alignment; no zone labels are involved."""
    out = []
    Fh = F.assign(h=np.asarray(signal, float))
    for p, d in Fh[Fh.h.notna()].groupby("participant"):
        tp = T[(T.participant == p) & (T.tvalid == 1)].dropna(subset=["tx"])
        rs = {}
        for v, dv in d.groupby("video"):
            tv = tp[tp.video == v].sort_values("t_ms")
            dv = dv.sort_values("epoch_ms")
            if len(tv) < 100 or len(dv) < 100:
                continue
            for lag in range(-span, span + 1, step):
                j = np.interp(dv.epoch_ms.values + lag, tv.t_ms.values, tv.tx.values)
                rs.setdefault(lag, []).append(np.corrcoef(dv.h.values, j)[0, 1])
        if not rs:
            continue
        mean_r = {l: float(np.nanmean(v)) for l, v in rs.items()}
        lag = max(mean_r, key=lambda l: mean_r[l])
        out.append({"participant": p, "setup": d.setup.iloc[0], "lag_ms": lag,
                    "r_at_lag": mean_r[lag], "r_at_zero": mean_r[0]})
    return pd.DataFrame(out)


def apply_sync(F, T, lags: dict):
    """Shift each participant's frame times by its lag and re-attach the nearest
    valid Tobii sample (within 20 ms), exactly as the extractor did."""
    F = F.copy()
    F["epoch_ms"] = F["epoch_ms"] + F["participant"].map(lags).fillna(0.0)
    F["tobii_x"] = np.nan; F["tobii_y"] = np.nan; F["tobii_dt_ms"] = np.nan
    for (p, v), idx in F.groupby(["participant", "video"]).groups.items():
        tv = T[(T.participant == p) & (T.video == v)].sort_values("t_ms")
        if len(tv) < 2:
            continue
        tt = tv.t_ms.values
        e = F.loc[idx, "epoch_ms"].values
        j = np.clip(np.searchsorted(tt, e), 1, len(tt) - 1)
        jn = np.where(np.abs(tt[j - 1] - e) <= np.abs(tt[j] - e), j - 1, j)
        dt = np.abs(tt[jn] - e)
        ok = (dt <= 20) & (tv.tvalid.values[jn] == 1)
        F.loc[idx, "tobii_x"] = np.where(ok, tv.tx.values[jn], np.nan)
        F.loc[idx, "tobii_y"] = np.where(ok, tv.ty.values[jn], np.nan)
        F.loc[idx, "tobii_dt_ms"] = dt
    return F


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--features", default="eott_features")
    ap.add_argument("--out", default="e1b_results")
    ap.add_argument("--sync", choices=["setup", "participant", "none"], default="setup",
                    help="clock correction between webcam and Tobii (v3). 'setup' = one median lag for Laptop and "
                         "one for PC (primary); 'participant' = each person's own lag; 'none' = v2 behaviour")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    rng = np.random.default_rng(SEED)

    F, T = load(a.features)
    # stage 1: out-of-fold predictions on the uncorrected data give a clean horizontal gaze
    # signal per frame; its best-matching lag against Tobii x is the clock offset
    print("stage 1: model on uncorrected clocks, to estimate the lag ...", flush=True)
    pred0 = fit_predict(F, F.tobii_x.notna().values & F.tobii_y.notna().values, n_folds=5)
    lags = estimate_lags(F, T, pred0[:, 0])
    lags.to_csv(os.path.join(a.out, "e1b_clock_lags.csv"), index=False)
    print(lags.groupby("setup").lag_ms.describe().round(0).to_string(), flush=True)
    if a.sync == "participant":
        F = apply_sync(F, T, dict(zip(lags.participant, lags.lag_ms)))
    elif a.sync == "setup":
        med = lags.groupby("setup").lag_ms.median().to_dict()
        F = apply_sync(F, T, {p: med[s_] for p, s_ in zip(lags.participant, lags.setup)})
    parts = sorted(F.participant.unique(), key=lambda s: int(s.split("_")[1]))
    qa = {"version": f"v3 (clock correction: {a.sync}; stability windows; model on all labelled frames)",
          "clock_lag_median_ms_by_setup": lags.groupby("setup").lag_ms.median().to_dict(),
          "n_participants": len(parts),
          "setups": F.groupby("setup").participant.nunique().to_dict(),
          "n_frames": int(len(F)),
          "face_detected": float(F.detected.mean()),
          "frames_with_tobii_label": float(F.tobii_x.notna().mean()),
          "median_tobii_dt_ms": float(F.tobii_dt_ms.median())}
    fps_by_video = {}
    for v, d in F.groupby(["participant", "video"]):
        span = (d.t_rel_ms.max() - d.t_rel_ms.min()) / 1000
        fps_by_video[v[1]] = (len(d) - 1) / span if span > 0 else 30.0
    qa["median_fps"] = float(np.median(list(fps_by_video.values())))

    # per-frame model on every frame with a face and a Tobii label (other participants only)
    train_mask = F.tobii_x.notna().values & F.tobii_y.notna().values
    qa["train_frames"] = int((train_mask & (F.detected.values == 1)).sum())
    print(json.dumps(qa, indent=2), flush=True)
    print("fitting per-frame model ...", flush=True)
    pred = fit_predict(F, train_mask)
    F["px"], F["py"] = pred[:, 0], pred[:, 1]

    lag = lag_check(F, T, pred)
    lag.to_csv(os.path.join(a.out, "e1b_lag_check.csv"), index=False)
    qa["lag_median_ms"] = float(lag.best_lag_ms.median())
    qa["lag_within_200ms_frac"] = float((lag.best_lag_ms.abs() <= 200).mean())
    qa["r_predx_tobiix_at_zero_median"] = float(lag.r_at_zero.median())

    # frame-level accuracy on all labelled frames (comparable with E1)
    fl = F[train_mask]
    frame_rows = []
    for (r, c) in GRIDS:
        ok = pd.Series(zone_of(fl.px, fl.py, r, c) == zone_of(fl.tobii_x, fl.tobii_y, r, c), index=fl.index)
        ps = ok.groupby(fl.participant).mean()
        lo, hi = boot_ci(ps.values, rng)
        su = ps.groupby(ps.index.map(F.groupby("participant").setup.first().to_dict())).mean()
        frame_rows.append({"grid": f"{r}x{c}", "acc": ps.mean(), "ci_lo": lo, "ci_hi": hi,
                           "acc_laptop": su.get("Laptop", np.nan), "acc_pc": su.get("PC", np.nan)})
    frame_acc = pd.DataFrame(frame_rows)

    setup_of = F.groupby("participant").setup.first().to_dict()
    rows, shared, wcount = [], [], []
    for disp in DISP_STD:
        for L in DWELLS_MS:
            W = make_windows(T, L, disp)
            npart = W.participant.nunique()
            wcount.append({"disp_std": disp, "dwell_ms": L, "windows": len(W), "participants": npart,
                           "median_per_participant": float(W.groupby("participant").size().median()) if len(W) else 0})
            print(f"disp {disp} dwell {L} ms: {len(W)} windows from {npart} participants", flush=True)
            if len(W) == 0:
                continue
            r_, s_ = score_windows(F, pred, W, L, fps_by_video, rng, setup_of)
            for x in r_: x.update(disp_std=disp, dwell_ms=L)
            for x in s_: x.update(disp_std=disp, dwell_ms=L)
            rows += r_; shared += s_
    acc, sh, wc = pd.DataFrame(rows), pd.DataFrame(shared), pd.DataFrame(wcount)
    acc.to_csv(os.path.join(a.out, "e1b_dwell_accuracy.csv"), index=False)
    frame_acc.to_csv(os.path.join(a.out, "e1b_frame_accuracy.csv"), index=False)
    sh.to_csv(os.path.join(a.out, "e1b_shared_error.csv"), index=False)
    wc.to_csv(os.path.join(a.out, "e1b_window_counts.csv"), index=False)
    with open(os.path.join(a.out, "e1b_qa.json"), "w") as f:
        json.dump(qa, f, indent=2)

    P = DISP_STD[0]
    L_ = ["# E1b (v3): dwell-level gaze-zone accuracy on EOTT (webcam video, Tobii ground truth)", "",
          f"Clock correction: **{a.sync}**. Webcam-to-Tobii lag estimated from stage-1 out-of-fold predictions, by setup "
          f"(median ms): {qa['clock_lag_median_ms_by_setup']}.", "",
          f"{qa['n_participants']} participants ({qa['setups']}), {qa['n_frames']} frames, "
          f"face detected in {qa['face_detected']:.1%}, median {qa['median_fps']:.1f} fps; "
          f"per-frame model trained on {qa['train_frames']} labelled frames (10-fold, grouped by participant).", "",
          f"**Timing check after correction:** best lag between predicted and Tobii x has median {qa['lag_median_ms']:.0f} ms; "
          f"{qa['lag_within_200ms_frac']:.0%} of participants within ±200 ms.", "",
          "## Windows found", "", "| stability (std) | dwell | windows | participants | median per participant |",
          "|---|---|---|---|---|"]
    for _, x in wc.iterrows():
        L_.append(f"| {x.disp_std} | {int(x.dwell_ms)} ms | {int(x.windows)} | {int(x.participants)} | {x.median_per_participant:.0f} |")
    L_ += ["", "## Single frames (all labelled frames; comparable with E1)", "",
           "| Grid | Accuracy | 95% CI | Laptop | PC |", "|---|---|---|---|---|"]
    for _, x in frame_acc.iterrows():
        L_.append(f"| {x.grid} | {x.acc:.3f} | {x.ci_lo:.3f}–{x.ci_hi:.3f} | {x.acc_laptop:.3f} | {x.acc_pc:.3f} |")
    for L in DWELLS_MS:
        s = acc[(acc.disp_std == P) & (acc.dwell_ms == L)] if len(acc) else acc
        if not len(s):
            continue
        L_ += ["", f"## Dwell {L} ms (primary stability threshold {P})", "",
               "| Grid | single frame | mean | median | vote | answered | mean, Laptop | mean, PC |",
               "|---|---|---|---|---|---|---|---|"]
        for g in [f"{r}x{c}" for r, c in GRIDS]:
            q = s[s.grid == g].set_index("method")
            L_.append(f"| {g} | {q.loc['single','acc']:.3f} | {q.loc['mean','acc']:.3f} "
                      f"({q.loc['mean','ci_lo']:.3f}–{q.loc['mean','ci_hi']:.3f}) | {q.loc['median','acc']:.3f} | "
                      f"{q.loc['vote','acc']:.3f} | {q.loc['mean','answered_frac']:.2f} | "
                      f"{q.loc['mean','acc_laptop']:.3f} | {q.loc['mean','acc_pc']:.3f} |")
    L_ += ["", "## Sensitivity: mean-of-dwell accuracy at other stability thresholds", "",
           "| stability | dwell | 1x2 | 1x3 | 2x2 | 3x3 |", "|---|---|---|---|---|---|"]
    for (d_, L), s in acc[acc.method == "mean"].groupby(["disp_std", "dwell_ms"]) if len(acc) else []:
        g = s.set_index("grid").acc
        L_.append(f"| {d_} | {L} ms | {g['1x2']:.3f} | {g['1x3']:.3f} | {g['2x2']:.3f} | {g['3x3']:.3f} |")
    L_ += ["", "## How much of the error is shared across a dwell", "",
           "ICC near 1: averaging cannot help. Near 0: averaging removes most of the error.", "",
           "| stability | dwell | axis | ICC | RMSE single frame | RMSE dwell mean | frames per window |",
           "|---|---|---|---|---|---|---|"]
    for _, x in sh.iterrows():
        L_.append(f"| {x.disp_std} | {x.dwell_ms} ms | {x.axis} | {x.shared_frac_icc:.2f} | {x.rmse_frame:.3f} | "
                  f"{x.rmse_window_mean:.3f} | {x.frames_per_window_mean:.1f} |")
    L_ += ["", "## QA", "", "```", json.dumps(qa, indent=2), "```"]
    with open(os.path.join(a.out, "E1b_REPORT.md"), "w") as f:
        f.write("\n".join(L_) + "\n")
    print("\n".join(L_))


if __name__ == "__main__":
    main()


### Main analysis: one clock correction per setup (Laptop, PC). About 25–35 min.

In [ ]:
!cd /content/e1b && python eott_analyze.py --sync setup --features "{WORK}/features" --out "{WORK}/results_v3/setup"

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{WORK}/results_v3/setup/E1b_REPORT.md').read()))

### Sensitivity check: each person's own clock correction. About 25–35 min.

In [ ]:
!cd /content/e1b && python eott_analyze.py --sync participant --features "{WORK}/features" --out "{WORK}/results_v3/participant"

In [ ]:
import shutil
shutil.make_archive('/content/E1b_results_v3', 'zip', f'{WORK}/results_v3')
from google.colab import files; files.download('/content/E1b_results_v3.zip')